In [ ]:
!nvidia-smi

Tue Dec 23 17:01:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')
# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
# print(data.read().decode("utf-8"))
lines = data.read().decode("utf-8").split('\n')[2:]

In [12]:
playlists = [s.rstrip().split() for s in lines if len(s.strip()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [13]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

In [14]:
!pip install gensim

In [15]:
from gensim.models import Word2Vec

# Create a Word2Vec model
model = Word2Vec(playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4)

In [16]:
song_id=2172
model.wv.most_similar(positive=str(song_id))

[('2849', 0.997228741645813),
 ('3167', 0.996863842010498),
 ('2104', 0.995850682258606),
 ('3117', 0.9956146478652954),
 ('2904', 0.9955976009368896),
 ('11517', 0.9953669905662537),
 ('3126', 0.9951125383377075),
 ('5549', 0.9950424432754517),
 ('5586', 0.9948574900627136),
 ('2640', 0.9946953654289246)]

In [17]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [19]:
import numpy as np
def print_recommendations(song_id):
    similar_songs = np.array(
    model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return songs_df.iloc[similar_songs]
# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
2849,Run To The Hills,Iron Maiden
3167,Unchained,Van Halen
2104,Life Won't Wait,Ozzy Osbourne
3117,Still Of The Night,Whitesnake
2904,Bark At The Moon,Ozzy Osbourne


# Song2Vec – Word2Vec Training Explained

This document explains **how Word2Vec is used to train song embeddings for recommendation**, combining motivation, intuition, and training mechanics in one place.

---

## Why We Train Song Embeddings

The goal is to learn a **dense vector for each song** such that:

* Songs that frequently appear together in playlists are close in vector space
* Songs that rarely co-occur are far apart

Once learned, these embeddings make similarity, recommendation, and retrieval computable using simple vector operations.

---

## Why Word2Vec Fits This Problem

Word2Vec is designed to learn representations from **co-occurrence in sequences**.

We use a direct analogy from NLP:

* Word → Song ID
* Sentence → Playlist
* Context window → Nearby songs in a playlist

The model is trained using **Skip-gram with Negative Sampling**, relying purely on playlist structure. No genre, artist, or metadata is used during training.

---

## What Happens During One Training Step

Consider a playlist fragment:

```
[Fade To Black, Unchained]
```

Assume `Fade To Black` is the center song and `Unchained` is a context song.

### 1. Dot product (raw similarity)

Each song has an embedding vector. The model computes:

```
score = v_fade · u_unchained
```

This measures how aligned the two vectors are.

* Positive → similar direction
* Near zero → weak relation
* Negative → dissimilar

---

### 2. Sigmoid (probability)

The score is passed through a sigmoid:

```
σ(score) = 1 / (1 + e^(−score))
```

This converts the score into a probability:

> “How likely do these two songs belong together?”

---

### 3. Loss and learning signal

* For real playlist pairs, the model increases this probability
* For randomly sampled songs (negative samples), the model decreases it

This creates two forces:

* Pull co-occurring songs together
* Push unrelated songs apart

---

### 4. Gradient descent (how vectors move)

Gradient descent updates embeddings so that:

* Dot products for real pairs increase
* Dot products for negative pairs decrease

Geometrically:

* Co-occurring songs rotate to point in similar directions
* Unrelated songs rotate away from each other

This process repeats millions of times across playlists.

---

## Why Clusters Emerge

Because playlists reflect real listening behavior:

* Similar songs co-occur often
* Dissimilar songs rarely do

Repeated updates cause the embedding space to self-organize:

* Dense regions correspond to genres or styles
* Distance reflects musical similarity

No labels are required for this structure to emerge.

---

## Why This Enables Recommendation

After training, each song ID maps to a vector that already encodes:

* Style and genre similarity
* Artist and era relationships
* User behavior patterns

Recommendation becomes a nearest-neighbor problem:

* Songs close in vector space are good recommendations

The same embeddings can be reused for:

* Song-to-song recommendation
* Clustering and catalog exploration
* Feature inputs to larger neural models
* Retrieval backends such as FAISS or RAG-style systems

This representation-first design is a core pattern in modern recommender systems.
